# Einheit 1: Was heißt Lernen bei einer Maschine

259.075 Data-integrated Algorithmic Design Processes II

Die Aufgabe des Semesters: aus den Kennwerten eines Bestandsgebäudes den Heizwärmebedarf
vorhersagen, und zwar schnell genug, dass man beim Entwerfen hundert Varianten durchrechnet
statt einer. Heute bauen wir den einfachsten Schätzer, den es gibt.

Die Gebäude sind echt: 1100 Wiener Häuser aus dem Layer GEBAEUDETYPOGD der Stadt Wien, mit
Bauperiode, Bautyp und Grundrissgeometrie, geholt von `data/fetch_ogd.py`.

Die Zielgröße ist es nicht. Der Heizwärmebedarf in dieser Tabelle ist gerechnet, nicht
gemessen: er kommt aus einer Formel, die den angenommenen Wand-U-Wert der Bauperiode und die
Kompaktheit verknüpft. Das Modell lernt heute also eine Rechenvorschrift nach, nicht das
Verhalten eines Hauses. Das ist Absicht, weil sich daran alles zeigen lässt, ohne dass
Messfehler dazwischenfunken. Ab Einheit 3 arbeiten wir mit gemessenen Daten.

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px

df = pd.read_csv("../data/gebaeude_wien.csv")
print(df.shape)
df.head()

## Die Punktwolke

Führ die Zelle aus. Jeder Punkt ist ein Haus. Waagrecht steht die Kompaktheit, also der
Umfang des Grundrisses geteilt durch die Wurzel aus seiner Fläche: ein Rundbau liegt bei 3,5,
ein langgezogener Zeilenbau deutlich darüber. Senkrecht steht der gerechnete Heizwärmebedarf.

Fahr über drei, vier Punkte und schau dir an, was für Häuser das sind. Frage, die du
beantworten können solltest, bevor du weiterliest: Warum liegen die Punkte nicht auf einer
Linie, obwohl der Bedarf aus einer Formel kommt?

In [ ]:
fig = px.scatter(df, x="kompaktheit", y="hwb", color="startjahr",
                 hover_data=["id", "bauperiode", "bautyp", "flaeche_m2", "u_wert"],
                 labels={"kompaktheit": "Kompaktheit (Umfang / Wurzel aus Fläche)",
                         "hwb": "Heizwärmebedarf, gerechnet [kWh/m2a]",
                         "startjahr": "ab Baujahr"},
                 color_continuous_scale="Viridis", height=520)
fig.update_traces(marker=dict(size=7, opacity=0.75))
fig.show()

Dieselben Häuser auf der Karte. Such dein Wohnhaus oder das Institut, klick es an und merk
dir die Nummer: mit diesem Haus arbeitest du am Ende der Einheit weiter.

In [ ]:
fig = px.scatter_map(df, lat="lat", lon="lon", color="hwb", size="flaeche_m2",
                     color_continuous_scale="Turbo", zoom=11, height=520,
                     hover_data=["id", "bauperiode", "kompaktheit", "u_wert"],
                     labels={"hwb": "Bedarf, gerechnet"},
                     map_style="carto-positron")
fig.show()

## Deine eigene Gerade

Führ die Zelle aus, dann zieh unten am Regler. Die Gerade `hwb = w * kompaktheit + b` wandert
mit, und in der Überschrift steht, wie groß dein Fehler gerade ist. Such den kleinsten Wert,
den du von Hand findest, und schreib ihn dir auf, wir vergleichen ihn gleich mit dem, was der
Rechner findet.

In [ ]:
def mse(w, b, x, y):
    return float(np.mean((w * x + b - y) ** 2))

x = df["kompaktheit"].to_numpy()
y = df["hwb"].to_numpy()
xs = np.linspace(x.min(), x.max(), 50)
b_fix = 110.0
ws = np.arange(0, 31, 1)

frames = [go.Frame(name=f"{w}",
                   data=[go.Scatter(x=xs, y=w * xs + b_fix, mode="lines",
                                    line=dict(width=4, color="crimson"))],
                   traces=[1],
                   layout=go.Layout(title=f"w = {w},  b = {b_fix:.0f},  Fehler = {mse(w, b_fix, x, y):.0f}"))
          for w in ws]

fig = go.Figure(
    data=[go.Scatter(x=x, y=y, mode="markers", name="Gebäude",
                     marker=dict(size=6, opacity=0.5, color="#4c78a8")),
          go.Scatter(x=xs, y=ws[0] * xs + b_fix, mode="lines", name="dein Modell",
                     line=dict(width=4, color="crimson"))],
    frames=frames)
fig.update_layout(
    height=560, xaxis_title="Kompaktheit", yaxis_title="Heizwärmebedarf, gerechnet",
    title=f"w = {ws[0]},  b = {b_fix:.0f},  Fehler = {mse(ws[0], b_fix, x, y):.0f}",
    sliders=[dict(active=0, currentvalue=dict(prefix="w = "),
                  steps=[dict(method="animate", label=f"{w}",
                              args=[[f"{w}"], dict(mode="immediate", frame=dict(duration=0, redraw=True))])
                         for w in ws])])
fig.show()

## Die Fehlerlandschaft

Derselbe Fehler, jetzt für alle Werte von `w` auf einmal. Das Tal ist das Ziel. Vergleich den
tiefsten Punkt der Kurve mit der Zahl, die du dir vorhin aufgeschrieben hast.

In [ ]:
w_grid = np.linspace(0, 30, 200)
losses = [mse(w, b_fix, x, y) for w in w_grid]

fig = go.Figure(go.Scatter(x=w_grid, y=losses, mode="lines", line=dict(width=3)))
fig.update_layout(height=420, xaxis_title="w", yaxis_title="mittlerer quadratischer Fehler",
                  title=f"Fehlerkurve über w, bei festem b = {b_fix:.0f}")
fig.show()

Weil das Modell zwei Parameter hat, ist die Kurve in Wahrheit eine Schale. Dreh sie mit der
Maus, bis du das Tal von der Seite siehst, und schau, wie schmal und schief es liegt. Genau
daran wird der Rechner gleich zu knabbern haben.

In [ ]:
W, B = np.meshgrid(np.linspace(0, 30, 60), np.linspace(60, 170, 60))
Z = np.array([[mse(w, b, x, y) for w in W[0]] for b in B[:, 0]])

fig = go.Figure(go.Surface(x=W[0], y=B[:, 0], z=Z, colorscale="Viridis", opacity=0.92))
fig.update_layout(height=620, title="Fehlerlandschaft über w und b",
                  scene=dict(xaxis_title="w", yaxis_title="b", zaxis_title="Fehler"))
fig.show()

## Bergab gehen

Die Steigung der Fehlerlandschaft hat einen Namen, sie heißt Gradient, und für dieses Modell
steht sie unten fertig da. Deine Aufgabe sind die zwei Zeilen in der Lücke: mach einen Schritt
entgegen der Steigung, einmal für `w` und einmal für `b`.

Wer die Herleitung des Gradienten sehen will, klappt sie im Angabetext auf. Geprüft wird sie
nicht.

In [ ]:
def gradient(w, b, x, y):
    fehler = w * x + b - y
    return float(2 * np.mean(fehler * x)), float(2 * np.mean(fehler))


def fit(x, y, lernrate=0.015, schritte=3000, w0=0.0, b0=None):
    w, b = w0, float(np.mean(y)) if b0 is None else b0
    verlauf = [(w, b, mse(w, b, x, y))]
    for _ in range(schritte):
        gw, gb = gradient(w, b, x, y)

        w -= lernrate * gw
        b -= lernrate * gb

        verlauf.append((w, b, mse(w, b, x, y)))
    return w, b, np.array(verlauf)


w_end, b_end, verlauf = fit(x, y)
print(f"w = {w_end:.1f}, b = {b_end:.1f}, Fehler = {verlauf[-1, 2]:.1f}")

In [ ]:
fig = go.Figure(go.Scatter(y=verlauf[:, 2], mode="lines", line=dict(width=3)))
fig.update_layout(height=380, xaxis_title="Schritt", yaxis_title="Fehler", yaxis_type="log",
                  title=f"Verlustkurve, Lernrate 0.015, Endfehler {verlauf[-1, 2]:.1f}")
fig.show()

## Der Abstieg als Film

Links die Höhenlinien der Fehlerlandschaft mit dem Weg, den die Parameter nehmen, rechts die
Gerade, die sich dabei in die Punktwolke legt. Play drücken und zusehen.

In [ ]:
from plotly.subplots import make_subplots

bilder = np.linspace(0, len(verlauf) - 1, 60).astype(int)
fig = make_subplots(rows=1, cols=2, subplot_titles=("Weg durch die Fehlerlandschaft",
                                                    "Modell in den Daten"))
fig.add_trace(go.Contour(x=W[0], y=B[:, 0], z=Z, colorscale="Viridis", showscale=False),
              row=1, col=1)
fig.add_trace(go.Scatter(x=[verlauf[0, 0]], y=[verlauf[0, 1]], mode="lines+markers",
                         marker=dict(size=7, color="crimson"), name="Parameter"), row=1, col=1)
fig.add_trace(go.Scatter(x=x, y=y, mode="markers", name="Gebäude",
                         marker=dict(size=5, opacity=0.4, color="#4c78a8")), row=1, col=2)
fig.add_trace(go.Scatter(x=xs, y=verlauf[0, 0] * xs + verlauf[0, 1], mode="lines",
                         line=dict(width=4, color="crimson"), name="Modell"), row=1, col=2)

fig.frames = [go.Frame(name=str(i),
                       data=[go.Scatter(x=verlauf[:i + 1, 0], y=verlauf[:i + 1, 1],
                                        mode="lines+markers",
                                        marker=dict(size=5, color="crimson")),
                             go.Scatter(x=xs, y=verlauf[i, 0] * xs + verlauf[i, 1],
                                        mode="lines", line=dict(width=4, color="crimson"))],
                       traces=[1, 3])
              for i in bilder]
fig.update_layout(height=560, showlegend=False,
                  updatemenus=[dict(type="buttons", x=0.02, y=1.15,
                                    buttons=[dict(label="Play", method="animate",
                                                  args=[None, dict(frame=dict(duration=60, redraw=True),
                                                                   fromcurrent=True)]),
                                             dict(label="Stop", method="animate",
                                                  args=[[None], dict(mode="immediate")])])])
fig.update_xaxes(title_text="w", row=1, col=1)
fig.update_yaxes(title_text="b", row=1, col=1)
fig.update_xaxes(title_text="Kompaktheit", row=1, col=2)
fig.update_yaxes(title_text="Heizwärmebedarf", row=1, col=2)
fig.show()

## Und jetzt kaputt machen

Hier änderst du nichts als eine Zahl: dreh die Lernrate hoch, bis die Kurve nach oben statt
nach unten geht. Für die Abgabe brauchst du beides, einen stabilen und einen divergenten
Lauf, dazu einen Satz, woran man den Unterschied in der Kurve sieht.

Zusatzfrage für die Schnellen: warum braucht der stabile Lauf dreitausend Schritte für eine
Gerade, die sich in einer Zeile ausrechnen ließe? Die Antwort liegt in der Form des Tals, das
du vorhin gedreht hast.

In [ ]:
_, _, wild = fit(x, y, lernrate=0.04, schritte=60)

fig = go.Figure(go.Scatter(y=wild[:, 2], mode="lines+markers", line=dict(width=3, color="crimson")))
fig.update_layout(height=380, xaxis_title="Schritt", yaxis_title="Fehler", yaxis_type="log",
                  title="Verlustkurve, Lernrate 0.04")
fig.show()

## Wo das Modell danebenliegt

Der Fehler bleibt stehen, und zwar nicht, weil die Schleife schlecht ist, sondern weil ein
Haus mehr hat als eine Grundrissform. Der angenommene U-Wert der Bauperiode steckt in der
Formel, das Modell kennt ihn nicht, und dieser Rest ist der Grund für Einheit 2.

Auf der Karte siehst du es: rot sind die Häuser, deren Bedarf das Modell überschätzt, blau die
unterschätzten. Such das Haus, das du dir vorhin gemerkt hast, und sag, warum es dort liegt,
wo es liegt.

In [ ]:
df["vorhersage"] = w_end * x + b_end
df["residuum"] = df["hwb"] - df["vorhersage"]

fig = px.scatter_map(df, lat="lat", lon="lon", color="residuum", zoom=11, height=560,
                     color_continuous_scale="RdBu", range_color=(-30, 30),
                     hover_data=["id", "bauperiode", "u_wert", "kompaktheit", "vorhersage", "hwb"],
                     map_style="carto-positron")
fig.show()

## Tests

Diese Zelle muss grün durchlaufen, bevor du abgibst. Sie rechnet selbst nach, was dein `fit`
zurückgibt, und vergleicht es mit der besten Geraden, die es für diese Daten gibt. Es sind
dieselben Tests, die auch die Bewertung verwendet, dort allerdings mit anderen Gebäuden.

In [ ]:
w_t, b_t, verlauf_t = fit(x, y)

eigener_fehler = float(np.mean((w_t * x + b_t - y) ** 2))
bestes_w, bestes_b = np.polyfit(x, y, 1)
bester_fehler = float(np.mean((bestes_w * x + bestes_b - y) ** 2))

assert np.isfinite([w_t, b_t]).all(), "die Parameter sind davongelaufen"
assert abs(eigener_fehler - verlauf_t[-1, 2]) < 1e-6, "der berichtete Fehler stimmt nicht mit den Parametern zusammen"
assert verlauf_t[-1, 2] < verlauf_t[0, 2], "der Fehler muss im Lauf kleiner werden"
assert eigener_fehler < 1.05 * bester_fehler, (
    f"noch zu weit weg: {eigener_fehler:.1f} gegen bestenfalls {bester_fehler:.1f}")
print(f"alles grün, Fehler {eigener_fehler:.2f}, bestenfalls {bester_fehler:.2f}")